# Experiment 3 — Multi-Seed Replication (MultipleNegativesRankingLoss)

**Reproducibility study, not a new experiment.** Runs the *existing* Experiment 3 pipeline unchanged, once per random seed, to measure run-to-run variance.

**The seed is the only variable.** Everything else is held fixed: encoder (all-MiniLM-L6-v2), MultipleNegativesRankingLoss (NO_DUPLICATES batch sampler), optimizer, scheduler, LR 2e-5, 3 epochs, batch size 32, checkpoints every 50 steps (all retained), validation-first selection by MRR@10 (tie-break HitRate@10), retrieval corpus, splits, preprocessing, and metrics.

**Scope of the seed.** The encoder is pretrained (deterministic init) and the (anchor, positive) pairs are fixed (derived from the same triplet file), so varying the seed varies *training stochasticity only*: batch composition/order and dropout. This isolates optimization variance — it deliberately does **not** resample the triplets, so both objectives see identical data in every run.

**Test-set discipline.** Every checkpoint is scored on validation only; the single validation-selected checkpoint is scored on test exactly once per seed.

**Only edit `REPO_URL`, set Runtime → GPU, then Runtime → Run all.**

In [ ]:
# --- The ONLY cell you edit ---
REPO_URL = "https://github.com/YOUR_USERNAME/distractor.git"  # <-- set this
SEEDS = [7, 17, 42, 123, 2025]
EXP = "exp3"          # label used in output filenames
OBJECTIVE = "mnrl"     # Experiment 3 objective — do not change in this notebook

In [ ]:
# --- Verify GPU ---
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > GPU"
print("GPU OK:", torch.cuda.get_device_name(0))

In [ ]:
# --- Clone repo and install pinned deps ---
import os
if not os.path.exists("distractor"):
    !git clone {REPO_URL} distractor
%cd distractor
%pip install -q -r requirements_gpu.txt
assert os.path.exists("datasets/train.csv"), "datasets/train.csv missing from repo"
print("Dataset present.")

In [ ]:
# --- Preprocessing: splits and triplets (generated once, shared by ALL seeds) ---
# Deliberately outside the seed loop: the data must be identical across seeds
# so that the seed is the only variable.
import os
if not os.path.exists("outputs/results/train_qdp.csv"):
    !python scripts/01_prepare_dataset.py
if not os.path.exists("outputs/triplets/train_triplets.jsonl"):
    !python scripts/03_create_triplets.py
print("Preprocessing ready.")

In [ ]:
# --- Baseline: computed ONCE and reused (it is deterministic; never retrained) ---
import os
if not os.path.exists("outputs/results/baseline_metrics.json"):
    !python scripts/02_baseline_retrieval.py
else:
    print("Baseline metrics already present — reusing.")

In [ ]:
# --- Multi-seed loop: train -> evaluate all checkpoints on VAL -> select -> TEST once ---
# Checkpoints are deleted after each seed is evaluated to bound Colab disk use
# (~15 checkpoints x ~90 MB x 5 seeds would otherwise exceed the quota).
import os, shutil, time

for seed in SEEDS:
    tag = f"{EXP}_seed{seed}_"
    model_dir = f"outputs/models/{EXP}_seed{seed}"
    if os.path.exists(f"outputs/results/{tag}final_test_results.csv"):
        print(f"=== seed {seed}: already done, skipping ===")
        continue
    print(f"\n{'='*70}\n=== SEED {seed} — training ({OBJECTIVE}) ===\n{'='*70}")
    t0 = time.time()
    !python scripts/train_gpu.py --objective {OBJECTIVE} --seed {seed} \
      --save-steps 50 --eval-steps 50 --keep-all-checkpoints \
      --output {model_dir}
    print(f"\n=== SEED {seed} — evaluating checkpoints (validation-first) ===")
    !python scripts/06_evaluate_checkpoints.py \
      --model-dir {model_dir} \
      --experiment-name "Experiment 3 (seed {seed})" \
      --skip-identity-check \
      --run-tag {tag} \
      --fig-dir-name {EXP}_seed{seed}
    shutil.rmtree(f"{model_dir}/checkpoints", ignore_errors=True)
    print(f"=== SEED {seed} done in {(time.time()-t0)/60:.1f} min (checkpoints freed) ===")

In [ ]:
# --- Aggregate: one row per seed ---
import pandas as pd, re, json

METRICS = ["HitRate@10", "Recall@10", "MRR@10", "nDCG@10",
           "HitRate@25", "Recall@25", "MRR@25", "nDCG@25"]
SEL_METRIC, SEL_TIEBREAK = "MRR@10", "HitRate@10"

rows = []
for seed in SEEDS:
    tag = f"{EXP}_seed{seed}_"
    val = pd.read_csv(f"outputs/results/{tag}checkpoint_metrics_val.csv")
    test = pd.read_csv(f"outputs/results/{tag}final_test_results.csv")
    # Re-derive the selection with the same rule used during evaluation,
    # then cross-check it against the step recorded in the test CSV.
    ck = val[val.training_step > 0]
    best = ck.sort_values([SEL_METRIC, SEL_TIEBREAK], ascending=False).iloc[0]
    best_step = int(best["training_step"])
    sel_col = [c for c in test.columns if c.startswith("selected")][0]
    recorded = int(re.search(r"step (\d+)", sel_col).group(1))
    assert recorded == best_step, f"seed {seed}: selection mismatch {recorded} vs {best_step}"
    row = {"seed": seed, "selected_checkpoint": f"checkpoint-{best_step}",
           "selected_step": best_step}
    for m in METRICS:
        row[f"val_{m}"] = float(best[m])
    tmap = dict(zip(test["metric"], test[sel_col]))
    for m in METRICS:
        row[f"test_{m}"] = float(tmap[m])
    rows.append(row)

seed_df = pd.DataFrame(rows)
seed_df.to_csv(f"outputs/results/{EXP}_seed_results.csv", index=False)
print(f"[Saved] outputs/results/{EXP}_seed_results.csv")
seed_df

In [ ]:
# --- Summary: mean / std / min / max per metric ---
num = seed_df.drop(columns=["seed", "selected_checkpoint"])
summary = num.agg(["mean", "std", "min", "max"]).T.reset_index()
summary.columns = ["metric", "mean", "std", "min", "max"]
summary["n_seeds"] = len(SEEDS)
summary.to_csv(f"outputs/results/{EXP}_summary.csv", index=False)
print(f"[Saved] outputs/results/{EXP}_summary.csv")
summary

In [ ]:
# --- Package results (includes per-query indices + splits so the analysis
#     notebook can run per-query significance tests) ---
import glob
files = (glob.glob(f"outputs/results/{EXP}_seed*_checkpoint_metrics_val.csv")
         + glob.glob(f"outputs/results/{EXP}_seed*_final_test_results.csv")
         + glob.glob(f"outputs/results/{EXP}_seed*_selected_indices.npy")
         + [f"outputs/results/{EXP}_seed_results.csv",
            f"outputs/results/{EXP}_summary.csv",
            "outputs/results/baseline_metrics.json",
            "outputs/results/baseline_indices.npy",
            "outputs/results/train_qdp.csv",
            "outputs/results/val_qdp.csv",
            "outputs/results/test_qdp.csv"])
files = [f for f in files if os.path.exists(f)]
!zip -q {EXP}_multiseed.zip {" ".join(files)}
print(f"Packaged {len(files)} files")
from google.colab import files as colab_files
colab_files.download(f"{EXP}_multiseed.zip")